# Bacpipe Tutorial
This notebook walks through the main workflows of the `bacpipe` library — from inspecting configuration and loading audio, through generating embeddings with built-in and custom models, to running the full pipeline and benchmarking classifier performance.

---
## 1. Setup & Configuration
Import necessary modules 

In [ ]:
# !BUG
# "kagglehub==0.3.13" is outdated, is it possible to change to "kagglehub>=0.3.13" ?
# ""scikit-learn==1.7.1"" is outdated, is it possible to change to ""scikit-learn>=1.7.1"?

# to run successfully the packages for jupyter notebook need to be installed:
# uv pip install ipykernel, ipython

from IPython.display import display
import os 
from pathlib import Path

# load the specific package
import bacpipe

Set the working directory to the repository root and clean the previous tests if needed/wanted

In [ ]:
# !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
# replace this with the path to the directory on your machine
# !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
# os.chdir('../..')
os.chdir('/media/haupert/data/mes_projets/_packages/bacpipe.git/bacpipe')

# Change the value of the key main_results_dir in the namespace bacpipe.settings to change the directory 
# where the results of the tutorials are stored. By default, it is set to './bacpipe_results'.
bacpipe.settings.main_results_dir = str(Path(bacpipe.settings.main_results_dir) / 'probing_a_model')

# !WARNING! the following code deletes the folder where the results of this tutorial is stored to be sure to start with a clean folder. 
# If you have important data in this folder, please comment it before running this code.
folder_path = bacpipe.settings.main_results_dir
if os.path.exists(folder_path):
    # Prompt the user
    user_input = input(f"Are you sure you want to delete '{folder_path}'? (y/n): ").lower().strip()

    if user_input == 'y':
        import shutil
        shutil.rmtree(folder_path) 
        print(f"Folder {folder_path} deleted.")
    else:
        print("Operation cancelled.")

else:
    print(f"Folder {folder_path} not found.")

Set global constants that will use through the notebook.

In [ ]:
MODEL_NAME = 'perch_v2'                  # name of the model to run. Supported models are in bacpipe.supported_models
AUDIO_DIR = 'bacpipe/tests/test_data'   # path to directory containing audio files

---
## 2. Probing Pipeline
Train a linear probe on top of frozen embeddings to evaluate how well the representations capture your specific ground truth annotations.

In [ ]:
embeds = bacpipe.run_pipeline_for_single_model(
    model_name=MODEL_NAME,                               # name of the model to run. Supported models are in bacpipe.supported_models
    audio_dir=AUDIO_DIR,                # path to directory containing audio files  
).embeddings(return_type='array')

# Run this function after computing the embeddings other it is no able to find the connection between embeddings and labels
gt = bacpipe.ground_truth_by_model(
    model=MODEL_NAME, 
    audio_dir=AUDIO_DIR, 
    annotations_filename='annotations.csv',
    single_label=False,
    overwrite=False
)

probe, label2idx, metrics = bacpipe.probing_pipeline(
    model_name=MODEL_NAME, 
    ground_truth=gt,
    embeds=embeds)


In [ ]:
print("======== display probe ==========")
display(type(probe))
print("======== display label2idx ======")
display(label2idx)
print("======== display metrics ========")
display(metrics)

---
## 3. Probe Inference
Load a previously trained linear probe and run inference to generate new predictions from your embeddings.

In [ ]:
# !BUG : 
# threshold should be optional as it is not necessary when return_binary_presence is False
# device should not be hardcoded to 'cuda' as it can cause errors on machines without cuda. 
# Propose a test before to set to 'cuda' or 'cpu' depending on the availability of cuda.

probe, label2idx = bacpipe.prepare_probe_inference(model=MODEL_NAME)

predictions = bacpipe.run_probe_inference(
    model=MODEL_NAME,
    linear_probe=probe,
    threshold=0.5,
    embeds=embeds,
    #device='cuda', 
    return_binary_presence=False
)

display(predictions)
